In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# ==============================
# 1. Dataset Path
# ==============================
dataset_path = r"D:\BrainTumorProject\dataset"

# ==============================
# 2. Device (GPU if available)
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# 3. Data Preprocessing + Augmentation
# ==============================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])

# ==============================
# 4. Load Dataset
# ==============================
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

print("Classes:", dataset.classes)

# ==============================
# 5. Train-Test Split
# ==============================
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ==============================
# 6. CNN Model
# ==============================
class BrainTumorCNN(nn.Module):

    def __init__(self):
        super(BrainTumorCNN, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*26*26,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x


model = BrainTumorCNN().to(device)

# ==============================
# 7. Loss Function + Optimizer
# ==============================
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==============================
# 8. Training
# ==============================
epochs = 25

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # ==============================
    # 9. Validation / Test Accuracy
    # ==============================
    model.eval()

    correct_test = 0
    total_test = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)

            predicted = (outputs > 0.5).float()

            correct_test += (predicted == labels).sum().item()
            total_test += labels.size(0)

    test_accuracy = 100 * correct_test / total_test

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Train Accuracy: {train_accuracy:.2f}% "
          f"Test Accuracy: {test_accuracy:.2f}%")

# ==============================
# 10. Final Evaluation
# ==============================
print("\nFinal Test Accuracy:", test_accuracy)

# ==============================
# 11. Save Model
# ==============================
torch.save(model.state_dict(), "brain_tumor_model.pth")

print("Model saved as brain_tumor_model.pth")

Using device: cpu
Classes: ['no', 'yes']
Epoch [1/25] Loss: 0.6794 Train Accuracy: 60.40% Test Accuracy: 52.94%
Epoch [2/25] Loss: 0.5856 Train Accuracy: 72.28% Test Accuracy: 80.39%
Epoch [3/25] Loss: 0.5544 Train Accuracy: 74.75% Test Accuracy: 60.78%
Epoch [4/25] Loss: 0.5251 Train Accuracy: 76.24% Test Accuracy: 78.43%
Epoch [5/25] Loss: 0.5089 Train Accuracy: 79.70% Test Accuracy: 72.55%
Epoch [6/25] Loss: 0.4464 Train Accuracy: 78.22% Test Accuracy: 72.55%
Epoch [7/25] Loss: 0.4230 Train Accuracy: 81.68% Test Accuracy: 60.78%
Epoch [8/25] Loss: 0.3910 Train Accuracy: 80.69% Test Accuracy: 68.63%
Epoch [9/25] Loss: 0.3982 Train Accuracy: 84.16% Test Accuracy: 68.63%
Epoch [10/25] Loss: 0.3185 Train Accuracy: 84.16% Test Accuracy: 78.43%
Epoch [11/25] Loss: 0.3264 Train Accuracy: 85.64% Test Accuracy: 80.39%
Epoch [12/25] Loss: 0.2725 Train Accuracy: 86.63% Test Accuracy: 76.47%
Epoch [13/25] Loss: 0.2681 Train Accuracy: 90.10% Test Accuracy: 76.47%
Epoch [14/25] Loss: 0.2776 Train